In [1]:
%matplotlib inline

In [2]:
from io import open
import unicodedata
import string
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

In [3]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
# device = 'cpu'
device

'mps'

In [4]:
!head rus.txt

Go.	Марш!	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #1159202 (shanghainese)
Go.	Иди.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #5898247 (marafon)
Go.	Идите.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #5898250 (marafon)
Hi.	Здравствуйте.	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #402127 (odexed)
Hi.	Привет!	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #466968 (katjka)
Hi.	Хай.	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #467233 (timsa)
Hi.	Здрасте.	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #3803577 (marafon)
Hi.	Здоро́во!	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #3854188 (marafon)
Hi.	Приветик!	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #7234283 (marafon)
Run!	Беги!	CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #1569978 (Biga)


In [5]:
SOS_token = 0
EOS_token = 1


class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [6]:
# Turn a Unicode string to plain ASCII, thanks to
# http://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizeString(s):
    # Lowercase, trim, and remove non-letter characters
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^а-яА-Яa-zA-Z.!?]+", r" ", s)
    return s

In [7]:
def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    # Read the file and split into lines
    lines = open('rus.txt', encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize
    pairs = [[normalizeString(s) for s in l.split('\t')][0:2] for l in lines]

    # Reverse pairs, make Lang instances
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

In [8]:
MAX_LENGTH = 10

# eng_prefixes = (
#     "i am ", "i m ",
#     "he is", "he s ",
#     "she is", "she s",
#     "you are", "you re ",
#     "we are", "we re ",
#     "they are", "they re "
# )


def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH # and \
        # p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [9]:
def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs


input_lang, output_lang, pairs = prepareData('eng', 'rus', True)
print(random.choice(pairs))

Reading lines...
Read 536124 sentence pairs
Trimmed to 441422 sentence pairs
Counting words...
Counted words:
rus 52680
eng 15824
['можешь воспользоваться моим .', 'you can use mine .']


## The Encoder

In [10]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        output, hidden = self.gru(output, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

## The Decoder

In [65]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super(AttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.attn = nn.Linear(self.hidden_size * 2, self.max_length)
        self.attn_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.gru = nn.GRU(self.hidden_size, self.hidden_size)
        self.out = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        embedded = self.dropout(embedded)

        attn_weights = F.softmax(
            self.attn(torch.cat((embedded[0], hidden[0]), 1)), dim=1)
        attn_applied = torch.bmm(attn_weights.unsqueeze(0),
                                 encoder_outputs.unsqueeze(0))

        output = torch.cat((embedded[0], attn_applied[0]), 1)
        output = self.attn_combine(output).unsqueeze(0)

        output = F.relu(output)
        output, hidden = self.gru(output, hidden)

        output = F.log_softmax(self.out(output[0]), dim=1)
        return output, hidden, attn_weights

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [11]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]


def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)


def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

In [12]:
teacher_forcing_ratio = 0.5


def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=MAX_LENGTH):
    encoder_hidden = encoder.initHidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

    loss = 0

    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(
            input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    decoder_input = torch.tensor([[SOS_token]], device=device)

    decoder_hidden = encoder_hidden

    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

    if use_teacher_forcing:
        # Teacher forcing: Feed the target as the next input
        for di in range(target_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  # Teacher forcing

    else:
        # Without teacher forcing: use its own predictions as the next input
        for di in range(target_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # detach from history as input

            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

In [13]:
import time
import math


def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)


def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [14]:
def trainIters(encoder, decoder, n_iters, print_every=1000, plot_every=100, learning_rate=0.01):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    training_pairs = [tensorsFromPair(random.choice(pairs))
                      for i in range(n_iters)]
    criterion = nn.NLLLoss()

    for iter in range(1, n_iters + 1):
        training_pair = training_pairs[iter - 1]
        input_tensor = training_pair[0]
        target_tensor = training_pair[1]

        loss = train(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if iter % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, iter / n_iters),
                                         iter, iter / n_iters * 100, print_loss_avg))

        if iter % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [15]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np


def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [16]:
def evaluate(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei],
                                                     encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        decoder_hidden = encoder_hidden

        decoded_words = []
        decoder_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            decoder_attentions[di] = decoder_attention.data
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words, decoder_attentions[:di + 1]

In [17]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, attentions = evaluate(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [20]:
hidden_size = 256
encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
attn_decoder1 = AttnDecoderRNN(hidden_size, output_lang.n_words, dropout_p=0.1).to(device)

trainIters(encoder1, attn_decoder1, 15000, print_every=5000)

3m 19s (- 6m 38s) (5000 33%) 4.4885
6m 32s (- 3m 16s) (10000 66%) 3.9212
9m 46s (- 0m 0s) (15000 100%) 3.6545


In [25]:
evaluateRandomly(encoder1, attn_decoder1)

> его выбрали капитаном .
= he was chosen captain .
< tom is to . . <EOS>

> я в нее влюбился .
= i fell in love with her .
< i was to in . . . <EOS>

> как поживаете ?
= how are you all ?
< how are you ? ? <EOS>

> я думаю что том честныи .
= i think tom is honest .
< i think that tom tom is . . <EOS>

> перестань мне врать .
= stop lying to me .
< i me . . <EOS>

> том обожал внуков .
= tom loved his grandchildren .
< tom was a . . . <EOS>

> сколько денег том накопил ?
= how much money did tom save ?
< how tom tom tom ? <EOS>

> том купил своим детям игрушек .
= tom bought his kids some toys .
< tom was a in a . . . <EOS>

> выпусти собаку пожалуиста .
= please let the dog out .
< please please . . . <EOS>

> индеики намного крупнее кур .
= turkeys are much larger than chickens .
< the can t be . . . <EOS>



## Внимание на скалярном произведении

### The Decoder

In [31]:
class DotAttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super(DotAttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.gru = nn.GRU(self.hidden_size * 2, self.hidden_size)  # без batch_first
        self.out = nn.Linear(self.hidden_size * 2, self.output_size)

    def forward(self, input, hidden, encoder_outputs):
        # input: (1) - текущий токен
        # hidden: (1, 1, hidden_size) - скрытое состояние декодера
        # encoder_outputs: (max_length, hidden_size) - все выходы энкодера
        
        embedded = self.embedding(input).view(1, 1, -1)  # (1, 1, hidden_size)
        embedded = self.dropout(embedded)
        
        # Вычисление весов внимания через скалярное произведение
        hidden_2d = hidden.squeeze(0)  # (1, hidden_size)
        
        # Скалярное произведение
        attn_energies = torch.matmul(encoder_outputs, hidden_2d.t())  # (max_length, 1)
        
        # Softmax
        attn_weights = F.softmax(attn_energies.t(), dim=1)  # (1, max_length)
        
        # Применяем веса внимания
        context = torch.mm(attn_weights, encoder_outputs)  # (1, hidden_size)
        context = context.unsqueeze(0)  # (1, 1, hidden_size)
        
        # Объединяем для входа в GRU
        gru_input = torch.cat((embedded, context), dim=2)  # (1, 1, hidden_size * 2)
        
        # GRU без batch_first ожидает вход (seq_len, batch, input_size)
        # У нас уже такой формат: (1, 1, hidden_size * 2)
        
        # Пропускаем через GRU
        output, hidden = self.gru(gru_input, hidden)  # output: (1, 1, hidden_size)
        
        # Объединяем для финального предсказания
        output = output.squeeze(0)  # (1, hidden_size)
        context = context.squeeze(0)  # (1, hidden_size)
        combined = torch.cat((output, context), dim=1)  # (1, hidden_size * 2)
        
        # Выходные вероятности
        output = F.log_softmax(self.out(combined), dim=1)
        
        return output, hidden, attn_weights

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [32]:
hidden_size = 256
encoder2 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
attn_decoder2 = DotAttnDecoderRNN(hidden_size, output_lang.n_words, dropout_p=0.1).to(device)

trainIters(encoder2, attn_decoder2, 5000, print_every=1000)

0m 42s (- 2m 48s) (1000 20%) 4.8747
1m 23s (- 2m 5s) (2000 40%) 4.4399
2m 5s (- 1m 23s) (3000 60%) 4.2089
2m 47s (- 0m 41s) (4000 80%) 4.0678
3m 29s (- 0m 0s) (5000 100%) 4.0127


In [33]:
evaluateRandomly(encoder2, attn_decoder2)

> том думал что мэри уже это сделала .
= tom thought mary had already done that .
< tom said that s what do that . <EOS>

> я знаю что том фотогеничен .
= i know tom is photogenic .
< i know tom was tom . <EOS>

> ваш ответ далеко не идеален .
= your answer is far from perfect .
< don t never . . . <EOS>

> ты совершенно бесполезен .
= you re completely useless .
< you re . . <EOS>

> поидем с нами пожалуиста .
= come with us please .
< please please . . <EOS>

> уже наверное слишком поздно .
= perhaps it s too late .
< my is is . <EOS>

> обещаете больше никому не рассказывать ?
= do you promise not to tell anyone else ?
< did don t do ? <EOS>

> чего мы здесь хотим ?
= what do we want here ?
< do we want here ? <EOS>

> зачем ты это купил ?
= why did you buy it ?
< did you think it ? <EOS>

> том отказался туда идти .
= tom refused to go there .
< tom was his . <EOS>



С использованием F.scaled_dot_product_attention

In [34]:
class ScaledDotAttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super(ScaledDotAttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.gru = nn.GRU(self.hidden_size * 2, self.hidden_size)
        self.out = nn.Linear(self.hidden_size * 2, self.output_size)
        
        # Проекционные слои для query, key, value (опционально, для улучшения качества)
        self.query_proj = nn.Linear(hidden_size, hidden_size)
        self.key_proj = nn.Linear(hidden_size, hidden_size)
        self.value_proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        embedded = self.dropout(embedded)
        
        # Проецируем для attention (опционально)
        query = self.query_proj(hidden.permute(1, 0, 2))  # (1, 1, hidden_size)
        key = self.key_proj(encoder_outputs.unsqueeze(0))  # (1, max_length, hidden_size)
        value = self.value_proj(encoder_outputs.unsqueeze(0))  # (1, max_length, hidden_size)
        
        # Scaled dot-product attention
        context = F.scaled_dot_product_attention(
            query, key, value,
            dropout_p=self.dropout_p if self.training else 0,
        )
        
        # Объединяем и подаем в GRU
        gru_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.gru(gru_input, hidden)
        
        # Финальное предсказание
        output = output.squeeze(0)
        context = context.squeeze(0)
        combined = torch.cat((output, context), dim=1)
        output = F.log_softmax(self.out(combined), dim=1)
        
        return output, hidden, None  # Возвращаем None вместо весов внимания

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [35]:
hidden_size = 256
encoder2 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
attn_decoder2 = ScaledDotAttnDecoderRNN(hidden_size, output_lang.n_words, dropout_p=0.1).to(device)

trainIters(encoder2, attn_decoder2, 5000, print_every=1000)

0m 46s (- 3m 6s) (1000 20%) 4.9639
1m 31s (- 2m 17s) (2000 40%) 4.4014
2m 17s (- 1m 31s) (3000 60%) 4.1687
3m 3s (- 0m 45s) (4000 80%) 4.0189
3m 50s (- 0m 0s) (5000 100%) 3.8829


### Внимание на основе MLP

### The Decoder

In [36]:
class SimpleMLPAttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super(SimpleMLPAttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.gru = nn.GRU(self.hidden_size * 2, self.hidden_size)
        self.out = nn.Linear(self.hidden_size * 2, self.output_size)
        
        # Единая MLP для внимания
        self.attn = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        embedded = self.dropout(embedded)
        
        # Подготовка скрытого состояния
        hidden_2d = hidden.squeeze(0)  # (1, hidden_size)
        
        # Для каждого выхода энкодера вычисляем энергию
        # encoder_outputs: (max_length, hidden_size)
        max_length = encoder_outputs.size(0)
        
        # Повторяем скрытое состояние для каждого выхода энкодера
        hidden_repeated = hidden_2d.repeat(max_length, 1)  # (max_length, hidden_size)
        
        # Конкатенируем с выходами энкодера
        combined = torch.cat((hidden_repeated, encoder_outputs), dim=1)  # (max_length, hidden_size * 2)
        
        # Пропускаем через MLP для получения энергий
        energies = self.attn(combined)  # (max_length, 1)
        energies = energies.t()  # (1, max_length)
        
        # Применяем softmax
        attn_weights = F.softmax(energies, dim=1)  # (1, max_length)
        
        # Вычисляем контекстный вектор
        context = torch.mm(attn_weights, encoder_outputs)  # (1, hidden_size)
        context = context.unsqueeze(0)  # (1, 1, hidden_size)
        
        # Дальше как обычно
        gru_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.gru(gru_input, hidden)
        
        output = output.squeeze(0)
        context = context.squeeze(0)
        combined = torch.cat((output, context), dim=1)
        output = F.log_softmax(self.out(combined), dim=1)
        
        return output, hidden, attn_weights

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [37]:
hidden_size = 256
encoder3 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
attn_decoder3 = SimpleMLPAttnDecoderRNN(hidden_size, output_lang.n_words, dropout_p=0.1).to(device)

trainIters(encoder3, attn_decoder3, 5000, print_every=1000)

0m 42s (- 2m 51s) (1000 20%) 4.9588
1m 24s (- 2m 7s) (2000 40%) 4.3830
2m 8s (- 1m 25s) (3000 60%) 4.1339
2m 51s (- 0m 42s) (4000 80%) 3.9682
3m 35s (- 0m 0s) (5000 100%) 3.8407


In [38]:
evaluateRandomly(encoder3, attn_decoder3)

> постараися высыпаться .
= make sure you get enough sleep .
< the s the . . <EOS>

> я вышел не на тои станции .
= i got off at the wrong station .
< i m not at at . <EOS>

> том сказал мне что ему жаль .
= tom told me he was sorry .
< tom told me me he . . <EOS>

> вы мне лжете ?
= are you lying to me ?
< are you you me me me ? <EOS>

> я туда не вернусь .
= i won t go back there .
< i m not at . <EOS>

> я была в шоке от увиденного .
= i was shocked to see that .
< i was the at . <EOS>

> этого вы добиваетесь ?
= is that your goal ?
< are you you you you ? ? <EOS>

> мне нужно с тобои вечером поговорить .
= i need to talk to you tonight .
< i need to to you . . . . <EOS>

> том вошел в пещеру .
= tom went into the cave .
< tom is at at . <EOS>

> мы сделаем это как нибудь в другои раз .
= we ll do this some other time .
< we ll that this to that . <EOS>



Выводы: после обучения на 5000 пар качество перевода оставляет желать лучшего.